# LAB 3 - BÀI 4: PHÂN LỚP IRIS VỚI K-FOLD CROSS VALIDATION

Sử dụng Neural Network để xây dựng mô hình phân lớp trên bộ dữ liệu Iris với k-fold cross validation

## a) Đọc bộ dữ liệu Iris từ sklearn

In [1]:
# Import các thư viện cần thiết
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Đọc bộ dữ liệu Iris
iris = load_iris()

print("Đã đọc bộ dữ liệu Iris thành công!")
print(f"Số lượng mẫu: {iris.data.shape[0]}")
print(f"Số lượng thuộc tính: {iris.data.shape[1]}")
print(f"Tên các lớp: {iris.target_names}")

Đã đọc bộ dữ liệu Iris thành công!
Số lượng mẫu: 150
Số lượng thuộc tính: 4
Tên các lớp: ['setosa' 'versicolor' 'virginica']


## b) Chuẩn hóa dữ liệu về đoạn [0,1]

In [2]:
# Gán X và Y
X = iris.data
Y = iris.target

# Chuẩn hóa X về đoạn [0,1]
X = X / X.max(axis=0)

print("Dữ liệu sau khi chuẩn hóa:")
print(f"Min values: {X.min(axis=0)}")
print(f"Max values: {X.max(axis=0)}")

Dữ liệu sau khi chuẩn hóa:
Min values: [0.5443038  0.45454545 0.14492754 0.04      ]
Max values: [1. 1. 1. 1.]


## c) Định nghĩa hàm create_model

In [3]:
def create_model():
    """
    Tạo và biên dịch mô hình Neural Network cho Iris
    Tương tự như Bài 1.h
    """
    model = Sequential([
        # Hidden layer 1: 10 neurons, activation ReLU
        Dense(10, activation='relu', input_shape=(4,)),
        
        # Hidden layer 2: 20 neurons, activation ReLU
        Dense(20, activation='relu'),
        
        # Output layer: 3 neurons (3 classes), activation Softmax
        Dense(3, activation='softmax')
    ])
    
    # Biên dịch mô hình
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Kiểm tra hàm create_model
print("Hàm create_model đã được định nghĩa!")
print("\nKiến trúc mô hình:")
test_model = create_model()
test_model.summary()

Hàm create_model đã được định nghĩa!

Kiến trúc mô hình:


c:\project\nop\Dataset\venv\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 10)             │            50 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 20)             │           220 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            63 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 333 (1.30 KB)

 Trainable params: 333 (1.30 KB)

 Non-trainable params: 0 (0.00 B)

## d) Huấn luyện mô hình bằng k-fold cross validation (k=10)

In [4]:
# Định nghĩa K-Fold Cross Validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)

# Mảng lưu trữ kết quả
models = []
accuracy_per_fold = []
precision_per_fold = []
recall_per_fold = []
f1_per_fold = []

# Mảng lưu trữ kết quả cho từng class
precision_per_class = [[], [], []]
recall_per_class = [[], [], []]
f1_per_class = [[], [], []]

print("="*70)
print("BẮT ĐẦU HUẤN LUYỆN VỚI 10-FOLD CROSS VALIDATION")
print("="*70)

fold_no = 1
for train_index, val_index in kfold.split(X):
    print(f"\n{'='*70}")
    print(f"FOLD {fold_no}/10")
    print(f"{'='*70}")
    
    # Tạo mô hình mới cho mỗi fold
    model = create_model()
    
    # Chia dữ liệu train và validation cho fold hiện tại
    X_train_fold = X[train_index]
    X_val_fold = X[val_index]
    y_train_fold = Y[train_index]
    y_val_fold = Y[val_index]
    
    # Huấn luyện mô hình (epochs=100, batch_size=1 như Bài 1.h)
    model.fit(
        X_train_fold, y_train_fold,
        epochs=100,
        batch_size=1,
        verbose=0  # Không hiển thị progress để đầu ra gọn hơn
    )
    
    # Lưu mô hình
    models.append(model)
    
    # Dự đoán trên tập validation
    y_pred_prob = model.predict(X_val_fold, verbose=0)
    y_pred = np.argmax(y_pred_prob, axis=1)
    
    # Tính các metrics
    accuracy = accuracy_score(y_val_fold, y_pred)
    precision_macro = precision_score(y_val_fold, y_pred, average='macro', zero_division=0)
    recall_macro = recall_score(y_val_fold, y_pred, average='macro', zero_division=0)
    f1_macro = f1_score(y_val_fold, y_pred, average='macro', zero_division=0)
    
    # Tính metrics cho từng class
    precision_per_class_fold = precision_score(y_val_fold, y_pred, average=None, zero_division=0)
    recall_per_class_fold = recall_score(y_val_fold, y_pred, average=None, zero_division=0)
    f1_per_class_fold = f1_score(y_val_fold, y_pred, average=None, zero_division=0)
    
    # Lưu kết quả
    accuracy_per_fold.append(accuracy)
    precision_per_fold.append(precision_macro)
    recall_per_fold.append(recall_macro)
    f1_per_fold.append(f1_macro)
    
    # Lưu kết quả cho từng class
    for i in range(3):
        precision_per_class[i].append(precision_per_class_fold[i])
        recall_per_class[i].append(recall_per_class_fold[i])
        f1_per_class[i].append(f1_per_class_fold[i])
    
    # In kết quả
    print(f"KẾT QUẢ FOLD {fold_no}:")
    print(f"  - Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"  - Precision: {precision_macro:.4f}")
    print(f"  - Recall:    {recall_macro:.4f}")
    print(f"  - F1-Score:  {f1_macro:.4f}")
    
    fold_no += 1

print(f"\n{'='*70}")
print("HOÀN THÀNH K-FOLD CROSS VALIDATION")
print(f"{'='*70}")

BẮT ĐẦU HUẤN LUYỆN VỚI 10-FOLD CROSS VALIDATION

FOLD 1/10
KẾT QUẢ FOLD 1:
  - Accuracy:  1.0000 (100.00%)
  - Precision: 1.0000
  - Recall:    1.0000
  - F1-Score:  1.0000

FOLD 2/10
KẾT QUẢ FOLD 1:
  - Accuracy:  1.0000 (100.00%)
  - Precision: 1.0000
  - Recall:    1.0000
  - F1-Score:  1.0000

FOLD 2/10


c:\project\nop\Dataset\venv\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


KẾT QUẢ FOLD 2:
  - Accuracy:  1.0000 (100.00%)
  - Precision: 1.0000
  - Recall:    1.0000
  - F1-Score:  1.0000

FOLD 3/10


c:\project\nop\Dataset\venv\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


KẾT QUẢ FOLD 3:
  - Accuracy:  1.0000 (100.00%)
  - Precision: 1.0000
  - Recall:    1.0000
  - F1-Score:  1.0000

FOLD 4/10


...

KẾT QUẢ FOLD 8:
  - Accuracy:  1.0000 (100.00%)
  - Precision: 1.0000
  - Recall:    1.0000
  - F1-Score:  1.0000

FOLD 9/10


c:\project\nop\Dataset\venv\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


KẾT QUẢ FOLD 9:
  - Accuracy:  1.0000 (100.00%)
  - Precision: 1.0000
  - Recall:    1.0000
  - F1-Score:  1.0000

FOLD 10/10


c:\project\nop\Dataset\venv\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


KẾT QUẢ FOLD 10:
  - Accuracy:  1.0000 (100.00%)
  - Precision: 1.0000
  - Recall:    1.0000
  - F1-Score:  1.0000

HOÀN THÀNH K-FOLD CROSS VALIDATION


## e) In ra Average Accuracy, Recall, Precision, F1-score

In [5]:
# Tính trung bình các metrics
avg_accuracy = np.mean(accuracy_per_fold)
avg_precision = np.mean(precision_per_fold)
avg_recall = np.mean(recall_per_fold)
avg_f1 = np.mean(f1_per_fold)

print("\n" + "="*70)
print("KẾT QUẢ TRUNG BÌNH SAU 10-FOLD CROSS VALIDATION")
print("="*70)
print(f"\nAverage Accuracy:  {avg_accuracy:.4f} ({avg_accuracy*100:.2f}%)")
print(f"Average Precision: {avg_precision:.4f}")
print(f"Average Recall:    {avg_recall:.4f}")
print(f"Average F1-Score:  {avg_f1:.4f}")

# Hiển thị bảng kết quả từng fold
results_df = pd.DataFrame({
    'Fold': range(1, 11),
    'Accuracy': accuracy_per_fold,
    'Precision': precision_per_fold,
    'Recall': recall_per_fold,
    'F1-Score': f1_per_fold
})

print("\n" + "="*70)
print("KẾT QUẢ CHI TIẾT TỪNG FOLD")
print("="*70)
print(results_df.to_string(index=False))


KẾT QUẢ TRUNG BÌNH SAU 10-FOLD CROSS VALIDATION

Average Accuracy:  0.9733 (97.33%)
Average Precision: 0.9752
Average Recall:    0.9767
Average F1-Score:  0.9720

KẾT QUẢ CHI TIẾT TỪNG FOLD
 Fold  Accuracy  Precision   Recall  F1-Score
    1  1.000000   1.000000 1.000000  1.000000
    2  1.000000   1.000000 1.000000  1.000000
    3  1.000000   1.000000 1.000000  1.000000
    4  0.933333   0.952381 0.933333  0.937322
    5  1.000000   1.000000 1.000000  1.000000
    6  0.866667   0.866667 0.888889  0.850000
    7  0.933333   0.933333 0.944444  0.932660
    8  1.000000   1.000000 1.000000  1.000000
    9  1.000000   1.000000 1.000000  1.000000
   10  1.000000   1.000000 1.000000  1.000000


## f) In ra Average Recall, Precision, F1-score cho từng class

In [6]:
# Tính trung bình cho từng class
avg_precision_per_class = [np.mean(precision_per_class[i]) for i in range(3)]
avg_recall_per_class = [np.mean(recall_per_class[i]) for i in range(3)]
avg_f1_per_class = [np.mean(f1_per_class[i]) for i in range(3)]

print("\n" + "="*70)
print("KẾT QUẢ TRUNG BÌNH CHO TỪNG CLASS")
print("="*70)

for i in range(3):
    print(f"\nClass {i} ({iris.target_names[i]}):")
    print(f"  - Average Precision: {avg_precision_per_class[i]:.4f}")
    print(f"  - Average Recall:    {avg_recall_per_class[i]:.4f}")
    print(f"  - Average F1-Score:  {avg_f1_per_class[i]:.4f}")

# Hiển thị bảng tổng hợp
class_results_df = pd.DataFrame({
    'Class': [f'{i} ({iris.target_names[i]})' for i in range(3)],
    'Avg Precision': avg_precision_per_class,
    'Avg Recall': avg_recall_per_class,
    'Avg F1-Score': avg_f1_per_class
})

print("\n" + "="*70)
print("BẢNG TỔNG HỢP KẾT QUẢ THEO CLASS")
print("="*70)
print(class_results_df.to_string(index=False))


KẾT QUẢ TRUNG BÌNH CHO TỪNG CLASS

Class 0 (setosa):
  - Average Precision: 1.0000
  - Average Recall:    1.0000
  - Average F1-Score:  1.0000

Class 1 (versicolor):
  - Average Precision: 0.9657
  - Average Recall:    0.9667
  - Average F1-Score:  0.9612

Class 2 (virginica):
  - Average Precision: 0.9600
  - Average Recall:    0.9633
  - Average F1-Score:  0.9548

BẢNG TỔNG HỢP KẾT QUẢ THEO CLASS
         Class  Avg Precision  Avg Recall  Avg F1-Score
    0 (setosa)       1.000000    1.000000      1.000000
1 (versicolor)       0.965714    0.966667      0.961197
 2 (virginica)       0.960000    0.963333      0.954798


## g) Sử dụng mô hình có F1-score tốt nhất để dự đoán 3 mẫu mới

In [7]:
# Tìm mô hình có F1-score tốt nhất
best_model_index = np.argmax(f1_per_fold)
best_model = models[best_model_index]
best_f1 = f1_per_fold[best_model_index]

print("="*70)
print("MÔ HÌNH CÓ F1-SCORE TỐT NHẤT")
print("="*70)
print(f"Fold: {best_model_index + 1}")
print(f"F1-Score: {best_f1:.4f}")
print(f"Accuracy: {accuracy_per_fold[best_model_index]:.4f}")
print(f"Precision: {precision_per_fold[best_model_index]:.4f}")
print(f"Recall: {recall_per_fold[best_model_index]:.4f}")

# 3 mẫu dữ liệu mới
new_samples = np.array([
    [6.2, 2.9, 4.3, 1.3],
    [5.1, 3.5, 1.4, 0.2],
    [7.3, 2.8, 6.4, 2.1]
])

print("\n" + "="*70)
print("DỰ ĐOÁN 3 MẪU DỮ LIỆU MỚI")
print("="*70)
print("\n3 mẫu dữ liệu:")
for i, sample in enumerate(new_samples):
    print(f"Mẫu {i+1}: {sample}")

# Chuẩn hóa dữ liệu mới (sử dụng max từ tập train gốc)
max_values = iris.data.max(axis=0)
new_samples_normalized = new_samples / max_values

# Dự đoán
predictions_prob = best_model.predict(new_samples_normalized, verbose=0)
predictions = np.argmax(predictions_prob, axis=1)

print("\nKẾT QUẢ DỰ ĐOÁN:")
print("="*70)
for i, (sample, pred) in enumerate(zip(new_samples, predictions)):
    confidence = np.max(predictions_prob[i]) * 100
    print(f"\nMẫu {i+1}: {sample}")
    print(f"  - Dự đoán: Class {pred} ({iris.target_names[pred]})")
    print(f"  - Độ tin cậy: {confidence:.2f}%")
    print(f"  - Xác suất cho từng class:")
    for j in range(3):
        print(f"      {iris.target_names[j]}: {predictions_prob[i][j]*100:.2f}%")

MÔ HÌNH CÓ F1-SCORE TỐT NHẤT
Fold: 1
F1-Score: 1.0000
Accuracy: 1.0000
Precision: 1.0000
Recall: 1.0000

DỰ ĐOÁN 3 MẪU DỮ LIỆU MỚI

3 mẫu dữ liệu:
Mẫu 1: [6.2 2.9 4.3 1.3]
Mẫu 2: [5.1 3.5 1.4 0.2]
Mẫu 3: [7.3 2.8 6.4 2.1]

KẾT QUẢ DỰ ĐOÁN:

Mẫu 1: [6.2 2.9 4.3 1.3]
  - Dự đoán: Class 1 (versicolor)
  - Độ tin cậy: 99.94%
  - Xác suất cho từng class:
      setosa: 0.00%
      versicolor: 99.94%
      virginica: 0.06%

Mẫu 2: [5.1 3.5 1.4 0.2]
  - Dự đoán: Class 0 (setosa)
  - Độ tin cậy: 100.00%
  - Xác suất cho từng class:
      setosa: 100.00%
      versicolor: 0.00%
      virginica: 0.00%

Mẫu 3: [7.3 2.8 6.4 2.1]
  - Dự đoán: Class 2 (virginica)
  - Độ tin cậy: 99.99%
  - Xác suất cho từng class:
      setosa: 0.00%
      versicolor: 0.01%
      virginica: 99.99%
